In [18]:
import joblib
import numpy as np
import pandas as pd

In [156]:
# Load
model = joblib.load('../model.pkl')
scaler = joblib.load('../scaler.pkl')

df_info = pd.read_csv('../data/processed/team-info.csv')
df_bracket = pd.read_csv('../2026_manual_bracket.csv').drop(columns=['Matchup'])
df_info = df_info[df_info['year'] == '2026']

# print(df_bracket)

df_out = pd.merge(df_bracket, df_info.add_prefix('team_a_'), left_on='team_a', right_on='team_a_team')
df_out = pd.merge(df_out, df_info.add_prefix('team_b_'), left_on='team_b', right_on='team_b_team')
df_out = df_out.rename(columns={'seed_A': 'team_a_seed', 'seed_B': 'team_b_seed'})

for col in ['seed', 'SOS', 'SRS', 'ORtg', 'DRtg'] :
    df_out[f'diff_{col}'] = df_out[f'team_a_{col}'].astype(float) - df_out[f'team_b_{col}'].astype(float)

# print(df_out[df_out['team_a_seed'] == 8].head(30))

for col in ['team', 'year', 'seed', 'SOS', 'SRS', 'ORtg', 'DRtg']:
    df_out = df_out.drop(columns=[f'team_a_{col}', f'team_b_{col}'])


# Predict
X = df_out[['diff_seed', 'diff_SOS', 'diff_SRS', 'diff_ORtg', 'diff_DRtg']]
X_scaled = scaler.transform(X)

df_out['team_a_win_pct'] = model.predict_proba(X_scaled)[:, 1]
df_out['predicted_winner'] = model.predict(X_scaled)
df_out

# prob = model.predict_proba(X_scaled)[0][1]
# pred = model.predict(X_scaled)[0]


# print(f"Predicted winner: Team {'A' if pred == 1 else 'B'}")
# print(f"Win probability: {prob:.1%}")

,team_a,team_b,diff_seed,diff_SOS,diff_SRS,diff_ORtg,diff_DRtg,team_a_win_pct,predicted_winner
0,duke,siena,-15.0,21.60,35.89,13.3,-7.7,0.996615,1
1,ohio-state,texas-christian,-1.0,2.55,3.31,7.5,5.9,0.662986,1
2,st-johns-ny,northern-iowa,-7.0,9.34,13.92,5.4,2.7,0.872943,1
3,kansas,california-baptist,-9.0,17.79,18.58,2.9,1.8,0.957264,1
4,louisville,south-florida,-5.0,8.23,10.12,1.7,0.4,0.810204,1
5,michigan-state,north-dakota-state,-11.0,18.70,20.94,1.2,1.6,0.958682,1
6,ucla,central-florida,-3.0,0.75,4.98,3.3,-3.3,0.625099,1
7,connecticut,furman,-13.0,16.67,26.16,3.1,-6.7,0.974099,1
8,michigan,howard,-15.0,26.46,37.02,10.6,-0.5,0.997188,1
9,georgia,saint-louis,-1.0,7.39,2.30,-0.6,9.7,0.609007,1
